# Unified Benchmark Notebook

This notebook combines circuit generation, simulation, and analysis. It saves QASM files and final statevectors to the `samples/` directory, logs timing information, and provides plotting utilities to visualize results.


In [ ]:
!pip install matplotlib pandas seaborn

| Routine type                     | Example                                    | Typical (p_H)         |
| -------------------------------- | ------------------------------------------ | --------------------- |
| State preparation                | Uniform superposition                      | **≈1.0**              |
| Fourier routines                 | Quantum Fourier Transform                  | **$0.02$–$0.10$**         |
| Search                           | Grover's algorithm                         | **$0.05$–$0.15$**         |
| Factoring / period finding       | Shor's algorithm                           | **$10^{-6}\text{--}10^{-3}$** |
| Variational circuits             | Variational Quantum Eigensolver            | **$0.01$–$0.05$**         |
| Optimization                     | Quantum Approximate Optimization Algorithm | **$0.01$–$0.03$**         |
| Random benchmark circuits        | Random Circuit Sampling                    | **$0.10$–$0.20$**         |
| Interference-heavy constructions | peaked circuits                            | **$0.30$–$0.50$**         |

In [ ]:
# Section 1: Import Libraries and Configure Environment
import os
import platform
import signal
import gc
import psutil, time, numpy as np, csv
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Qiskit and DDSIM
import qiskit
from qiskit_aer import StatevectorSimulator
import qiskit.qasm2, qiskit.qasm3
from mqt import ddsim

# PolyQ modules
from PolyQ.utils.random_circuit_generator_universal import *
from PolyQ.engine import *

# enable autoreload
%load_ext autoreload
%autoreload 2

# set seaborn theme
sns.set_theme(style="whitegrid")

In [ ]:
# Section 2: Set Up Sample Directory and CSV Logging
os.makedirs('samples/circuits', exist_ok=True)
os.makedirs('Results/demo', exist_ok=True)
results_csv = 'samples/benchmark_results.csv'
if not os.path.exists(results_csv):
    with open(results_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['n','h','d','g','t','h_prob',
                         'cpu_time_poly','wall_time_poly',
                         'cpu_time_ddsim','wall_time_ddsim',
                         'cpu_time_aer','wall_time_aer'])

In [ ]:
# Section 3: Utility Functions: Circuit Generation and Formatting

def get_ketgpt_circ(n: int, h_prob: float = 0.1):
    """Stub for obtaining a circuit from KetGPT.

    In a real workflow this would call the KetGPT library or service to
    generate a circuit specification.  For now we fall back on the
    random generator using h_prob as a hint.
    """
    # TODO: replace with actual KetGPT invocation, e.g.:
    #     return ketgpt.generate_circuit(num_qubits=n, h_probability=h_prob)
    qc, qr, seed = random_circ_h_const(n, int(h_prob*n), h_prob)
    return qc, qr, seed


def format_complex(c):
    real = 0.0 if abs(c.real) < 1e-6 else round(c.real, 3)
    imag = 0.0 if abs(c.imag) < 1e-6 else round(c.imag, 3)
    return f"{real:+.3f}{imag:+.3f}j"


In [ ]:
# Section 4: Simulation Functions for PolyQ, Aer, DDSIM

def get_stvec_poly(qc, n, t, initial_state):
    terms, wire_array, max_new_var = create_poly(qc, n)
    assert t == max_new_var, "Value of 't' != 'max_new_var' from create_poly"
    ovs = [j[-1] for j in wire_array]
    ttb = get_truthtable_no_ivs(terms, n, t, initial_state)
    stvec = get_statevector_file(ttb, n, t, ovs)
    del ttb, terms, wire_array, max_new_var
    return stvec


def get_stvec_ddsim(qc):
    backend = ddsim.DDSIMProvider().get_backend("statevector_simulator")
    job = backend.run(qc)
    result = job.result()
    return result.get_statevector()


def get_stvec_aer(qc):
    backend = StatevectorSimulator()
    res = backend.run(qc).result()
    return res.get_statevector()

# one-step wrapper if desired
from PolyQ.simulation import simulate

def simulate_and_format(qc):
    st = simulate(qc)
    return [format_complex(c) for c in st]


In [ ]:
# Section 5: Timing and Timeout Helpers

def get_time_poly(qc, n, t, initial_state):
    start = psutil.Process().cpu_times()
    t0 = time.time()
    if n == t:
        st = np.zeros(1,dtype=complex)
    else:
        st = get_stvec_poly(qc, n, t, initial_state)
    end = psutil.Process().cpu_times()
    t1 = time.time()
    cpu = (end.user - start.user) + (end.system - start.system)
    return st, cpu, t1-t0


def get_time_ddsim(qc):
    start = psutil.Process().cpu_times()
    t0 = time.time()
    st = get_stvec_ddsim(qc)
    end = psutil.Process().cpu_times()
    t1 = time.time()
    cpu = (end.user - start.user) + (end.system - start.system)
    return st, cpu, t1-t0


def get_time_aer(qc):
    start = psutil.Process().cpu_times()
    t0 = time.time()
    st = get_stvec_aer(qc)
    end = psutil.Process().cpu_times()
    t1 = time.time()
    cpu = (end.user - start.user) + (end.system - start.system)
    return st, cpu, t1-t0

# timeout helpers

def timeout_handler(signum, frame):
    raise TimeoutError("Process exceeded time limit")


def execute_with_timeout(timeout, func, *args):
    stop_flag = False
    result = None
    if platform.system() != 'Windows' and hasattr(signal, 'SIGALRM'):
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)
        try:
            result = func(*args)
        except TimeoutError:
            stop_flag = True
        finally:
            signal.alarm(0)
    else:
        result_container = {'result': None, 'exception': None}
        def target():
            try:
                result_container['result'] = func(*args)
            except Exception as e:
                result_container['exception'] = e
        thread = threading.Thread(target=target)
        thread.daemon = True
        thread.start()
        thread.join(timeout)
        if thread.is_alive() or result_container['exception']:
            stop_flag = True
        else:
            result = result_container['result']
    return result, stop_flag


In [ ]:
# Section 6: Saving QASM and Statevectors to Samples

def save_qasm(qc, name_prefix):
    qasm2 = qiskit.qasm2.dumps(qc)
    qasm3 = qiskit.qasm3.dumps(qc)
    with open(f'samples/circuits/{name_prefix}.qasm2','w') as f:
        f.write(qasm2)
    with open(f'samples/circuits/{name_prefix}.qasm3','w') as f:
        f.write(qasm3)


def save_statevector(st, name_prefix):
    # st can be numpy array or list
    arr = np.asarray(st)
    np.savetxt(f'samples/{name_prefix}_statevec.txt', arr.astype(complex), fmt='%s')
    # also csv
    np.savetxt(f'samples/{name_prefix}_statevec.csv', arr.astype(complex), fmt='%s', delimiter=',')


In [ ]:
# Section 7: Benchmark Loop: Generate Circuits, Run Simulators, Save Results

# increase qubit range and sample multiple h_prob values
timeout = 3  # seconds

for h_prob in np.arange(0.01, 0.101, 0.01):
    for n in range(3, 9):  # extended qubit range
        # obtain circuit from KetGPT (placeholder stub)
        qc, qr, seed = get_ketgpt_circ(n, h_prob)

        # transpile to Clifford+T basis
        qc = qiskit.transpile(qc,
                               basis_gates=['h','s','sdg','t','tdg','cx','cz','ccx','ccz'],
                               optimization_level=3)

        n_eff = qc.width()
        h_eff = list(instr.operation.name for instr in qc.data).count('h')
        d = qc.depth()
        g = gate_counts(qc)
        t = n_eff + h_eff
        print(f"circuit n={n_eff}, h={h_eff}, d={d}, g={g}, t={t}, h_prob={h_prob} (ketgpt)")
        initial_state = [0]*n_eff

        # save qasm with prefix
        prefix = f'ketgpt_n{n_eff}_h{h_eff}_hp{h_prob}'
        save_qasm(qc, prefix)

        # run simulators with timeout
        st_poly, cpu_poly, wall_poly = (None,-1,-1)
        st_aer, cpu_aer, wall_aer = (None,-1,-1)
        st_ddsim, cpu_ddsim, wall_ddsim = (None,-1,-1)

        if t != n_eff:
            res, stop = execute_with_timeout(timeout, get_time_poly, qc, n_eff, t, initial_state)
            if not stop:
                st_poly, cpu_poly, wall_poly = res
                save_statevector(st_poly, prefix + '_poly')
        
        res, stop = execute_with_timeout(timeout, get_time_aer, qc)
        if not stop:
            st_aer, cpu_aer, wall_aer = res
            save_statevector(st_aer, prefix + '_aer')

        res, stop = execute_with_timeout(timeout, get_time_ddsim, qc)
        if not stop:
            st_ddsim, cpu_ddsim, wall_ddsim = res
            save_statevector(st_ddsim, prefix + '_ddsim')

        # append to CSV
        with open(results_csv, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([n_eff, h_eff, d, g, t, h_prob,
                             cpu_poly, wall_poly,
                             cpu_ddsim, wall_ddsim,
                             cpu_aer, wall_aer])

        # cleanup
        del qc, qr, seed, initial_state, st_poly, st_aer, st_ddsim
        gc.collect()

In [ ]:
# Section 8: Plotting Graphs from Results CSV

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

# qubits vs wall time for each simulator
for sim in ['wall_time_poly','wall_time_aer','wall_time_ddsim']:
    plt.figure()
    sns.lineplot(data=df, x='n', y=sim, marker='o')
    plt.yscale('log')
    plt.title(sim + ' vs n')
    plt.show()

# h gates vs wall time
for sim in ['wall_time_poly','wall_time_aer','wall_time_ddsim']:
    plt.figure()
    sns.lineplot(data=df, x='h', y=sim, marker='o')
    plt.yscale('log')
    plt.title(sim + ' vs h')
    plt.show()

# comparison overlay
plt.figure(figsize=(10,6))
sns.lineplot(data=df, x='n', y='wall_time_aer', marker='o', label='Aer')
sns.lineplot(data=df, x='n', y='wall_time_ddsim', marker='o', label='DDSIM')
sns.lineplot(data=df, x='n', y='wall_time_poly', marker='o', label='PolyQ')
plt.yscale('log')
plt.title('Wall time comparison')
plt.legend()
plt.show()


In [ ]:
# Section 9: Cleanup Artifacts

def cleanup_artifacts():
    """Remove generated files and directories created during benchmarking."""
    import shutil

    paths = ['samples', 'Results', results_csv]
    for p in paths:
        if os.path.isdir(p):
            shutil.rmtree(p)
        elif os.path.isfile(p):
            os.remove(p)

    print("Cleanup complete: removed samples, Results/demo, and benchmark CSV.")

# call the function if desired
cleanup_artifacts()